In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

results = pd.read_csv(
    PROJECT_ROOT / "data" / "evaluation_results.csv"
)

print("Total evaluation questions:", len(results))
results.head()

Total evaluation questions: 10


,question,expected_answer,generated_answer,source_found,top_source,top_page,similarity
0,What was Apple's total net sales in 2025?,"$416,161 million","In 2025, Apple's total net sales were $416,161...",True,_10-K-2025-As-Filed.pdf,26.0,0.738629
1,What were Apple's iPhone net sales in 2025?,"$209,586 million","In 2025, Apple's iPhone net sales were **$209,...",True,10-Q4-2024-As-Filed.pdf,26.0,0.767434
2,What were Apple's Mac net sales in 2025?,"$33,708 million","In 2025, Apple's Mac net sales were $33,708 mi...",True,_10-K-2025-As-Filed.pdf,26.0,0.710457
3,What were Apple's Services net sales in 2025?,"$109,158 million","In 2025, Apple's Services net sales were $109,...",True,_10-K-2025-As-Filed.pdf,26.0,0.749651
4,What was Apple's total net sales in 2024?,"$391,035 million",ERROR,False,NaN,NaN,NaN


In [2]:
total_questions = len(results)

successful_answers = (
    results["generated_answer"] != "ERROR"
).sum()

retrieved_sources = (
    results["source_found"] == True
).sum()

api_errors = (
    results["generated_answer"] == "ERROR"
).sum()

retrieval_rate = retrieved_sources / total_questions * 100
generation_rate = successful_answers / total_questions * 100

print("===== RAG EVALUATION SUMMARY =====")
print("Total Questions:", total_questions)
print("Successful Answers:", successful_answers)
print("Retrieved Sources:", retrieved_sources)
print(f"Retrieval Success Rate: {retrieval_rate:.1f}%")
print(f"Answer Generation Rate: {generation_rate:.1f}%")
print("API/Generation Errors:", api_errors)

===== RAG EVALUATION SUMMARY =====
Total Questions: 10
Successful Answers: 8
Retrieved Sources: 8
Retrieval Success Rate: 80.0%
Answer Generation Rate: 80.0%
API/Generation Errors: 2


In [3]:
def check_answer(row):
    if row["generated_answer"] == "ERROR":
        return "API_ERROR"

    expected = str(row["expected_answer"]).replace(",", "").replace("$", "").strip()
    generated = str(row["generated_answer"]).replace(",", "").replace("$", "").strip()

    if expected in generated:
        return "CORRECT"

    return "REVIEW"


results["correctness"] = results.apply(check_answer, axis=1)

print("===== ANSWER CORRECTNESS =====")
print(results["correctness"].value_counts())

results[[
    "question",
    "expected_answer",
    "correctness"
]]

===== ANSWER CORRECTNESS =====
correctness
CORRECT      8
API_ERROR    2
Name: count, dtype: int64


,question,expected_answer,correctness
0,What was Apple's total net sales in 2025?,"$416,161 million",CORRECT
1,What were Apple's iPhone net sales in 2025?,"$209,586 million",CORRECT
2,What were Apple's Mac net sales in 2025?,"$33,708 million",CORRECT
3,What were Apple's Services net sales in 2025?,"$109,158 million",CORRECT
4,What was Apple's total net sales in 2024?,"$391,035 million",API_ERROR
5,What were Apple's iPhone net sales in 2024?,"$201,183 million",CORRECT
6,What were Apple's Services net sales in 2024?,"$96,169 million",CORRECT
7,What was Apple's total net sales in 2023?,"$383,285 million",CORRECT
8,What were Apple's iPhone net sales in 2023?,"$200,583 million",CORRECT
9,What were Apple's Services net sales in 2023?,"$85,200 million",API_ERROR


## Evaluation Conclusion

The evaluation dataset contains 10 financial questions covering Apple's 2023, 2024, and 2025 financial data.

Results:
- Total questions: 10
- Correct generated answers: 8
- API/generation errors: 2
- Correctness among successfully generated answers: 8/8 (100%)
- Overall successful generation rate: 8/10 (80%)

The two API errors were caused by LLM API quota limitations during evaluation. They are reported separately from answer correctness rather than being treated as incorrect financial answers.

The successful responses matched the expected financial values and were supported by retrieved document sources.